# Prerequisite 0: Prepare Your Python Kernel

1. Use **Select Kernel** (top right) to pick **any Python 3.10+ kernel**: the bundled Python, conda, or a venv all work.
2. Run the cell below. It installs the lab packages (`snowflake-connector-python`, `pandas`, `matplotlib`) into that kernel. It uses pip if the environment has it, and falls back to [uv](https://docs.astral.sh/uv/) for environments that have no pip, such as venvs created by uv.

You only need to run it once per kernel. Running it again is safe; it skips anything already installed.

**Optional, for a clean environment:** in a terminal at the repo root, run `uv sync`, then select the `.venv` interpreter under **Select Kernel → Python Environments**.

In [ ]:
# Prerequisite 0: install the lab packages into whatever kernel is selected.
import importlib, importlib.util, shutil, subprocess, sys

PACKAGES = {"snowflake-connector-python": "snowflake.connector", "pandas": "pandas", "matplotlib": "matplotlib"}

def installed(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False

print(f"Kernel Python: {sys.executable} ({sys.version.split()[0]})")
if sys.version_info < (3, 10):
    raise SystemExit("This lab needs Python 3.10+. Select a newer kernel and re-run.")

missing = [pkg for pkg, mod in PACKAGES.items() if not installed(mod)]
if missing:
    print("Installing:", ", ".join(missing))
    has_pip = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True).returncode == 0
    if has_pip:
        cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    elif shutil.which("uv"):
        cmd = ["uv", "pip", "install", "-q", "--python", sys.executable, *missing]
    else:
        raise SystemExit("This kernel has no pip and uv is not installed. Install uv (brew install uv, or "
                         "https://docs.astral.sh/uv/) or pick a different kernel, then re-run.")
    subprocess.run(cmd, check=True)
    importlib.invalidate_caches()

still_missing = [pkg for pkg, mod in PACKAGES.items() if not installed(mod)]
if still_missing:
    raise SystemExit(f"Still missing {still_missing}. Restart the kernel and re-run this cell.")
print("Packages ready. Continue with the next prerequisite.")

# Prerequisite: Set Up Your SQL Environment

Run the cell below once before the lab, and again any time the notebook kernel restarts.

This notebook runs on a local Python kernel, which cannot run SQL by itself. The cell below sets that up:

1. **Uses the packages from Prerequisite 0** (`snowflake-connector-python` and pandas) in your selected kernel.
2. **Connects to your lab account** by name. Your default connection may point to a different account, so the cell names the lab account explicitly.
3. **Sets the lab context**:
   - **Role `ACCOUNTADMIN`**: owns `SEAT_SWEEPER_LAB` and can create the dynamic tables, views and app objects. Your session otherwise starts as `ATTENDEE_ROLE`, which cannot.
   - **Warehouse `SEAT_SWEEPER_WH`**: the lab's X-Small warehouse, which auto-suspends after 60 seconds idle.
   - **Database `SEAT_SWEEPER_LAB`, schema `APP`**: where the pipeline and app objects live.
4. **Adds a `%%sql` helper** so the SQL cells in the lab run on Snowflake.

The output table shows your active context. Confirm it reads `ACCOUNTADMIN`, `SEAT_SWEEPER_WH`, `SEAT_SWEEPER_LAB` and `APP` before you continue to `lab_instructions.ipynb`.

> **Note:** This changes only your session. It creates and grants nothing in Snowflake.

### How to write SQL cells in this lab

Start every SQL cell with `%%sql` on its own first line. Everything below it runs on Snowflake. You can put several statements in one cell; the last result is shown.

**Comments:** use `--` for a single line or `/* ... */` for a block, placed **after** the `%%sql` line. Never use `#`, which is not valid in Snowflake SQL.

```sql
%%sql
-- This is a single-line SQL comment
/* This is a
   block comment */
SELECT CURRENT_ROLE();
```

> **If you see `SyntaxError: invalid syntax` pointing at a `--` line**, the comment is fine. The cell is missing `%%sql` on its first line, so Python tried to run it. Add `%%sql` and re-run it.
>
> **If you see `UsageError: Cell magic %%sql not found`**, the kernel restarted. Re-run this prerequisite cell.

In [ ]:
# Setup: connect this notebook to your lab account and enable %%sql cells.
# Run once per kernel session (again after any kernel restart).
# Packages come from the uv environment or Prerequisite 0.

import snowflake.connector
from IPython import get_ipython
from IPython.display import display
import pandas as pd
pd.set_option("display.max_colwidth", None)  # show full text (log messages, queries) instead of "..."

# The lab account's connection in ~/.snowflake/connections.toml.
# Your default connection may point to a different account, so name it explicitly.
CONNECTION = "REPLACE_WITH_YOUR_NAMED_CONNECTION"  # from: snow connection list

conn = snowflake.connector.connect(connection_name=CONNECTION)
for stmt in ("USE ROLE ACCOUNTADMIN", "USE WAREHOUSE SEAT_SWEEPER_WH",
             "USE DATABASE SEAT_SWEEPER_LAB", "USE SCHEMA APP"):
    conn.cursor().execute(stmt)

def sql(line, cell):
    """Run every statement in the cell on the lab connection; show the last result."""
    cur = None
    for cur in conn.execute_string(cell, remove_comments=True):
        pass
    if cur is not None and cur.description:
        display(pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description]))

get_ipython().register_magic_function(sql, magic_kind="cell", magic_name="sql")

display(pd.DataFrame(conn.cursor().execute(
    "SELECT CURRENT_ACCOUNT_NAME() AS account, CURRENT_USER() AS user_name, CURRENT_ROLE() AS role, "
    "CURRENT_WAREHOUSE() AS warehouse, CURRENT_DATABASE() AS database, CURRENT_SCHEMA() AS schema"
).fetchall(), columns=["ACCOUNT", "USER_NAME", "ROLE", "WAREHOUSE", "DATABASE", "SCHEMA"]))